# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [4]:
# Write your code below.

%load_ext dotenv
%dotenv 

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import dask.dataframe as dd
import pandas as pd
import sys
import os
from glob import glob

PRICE_DATA = os.getenv('PRICE_DATA')

print(PRICE_DATA)

# Do I need to save this as PRICE_DATA? 

# Do I need these steps?:

# if os.path.exists(PRICE_DATA):
#     shutil.rmtree(PRICE_DATA)
# dd_rets.to_parquet(PRICE_DATA, overwrite = True)



../../05_src/data/prices/


In [44]:
# Write your code below.

#parquet_files = glob('../../05_src/data/prices/**/*.parquet', recursive= True)
#parquet_files = glob('PRICE_DATA', recursive= True) # join price data directory
#parquet_files

#stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))


#parquet_files = os.path.join(PRICE_DATA, "**/*.parquet")

#parquet_files = glob(os.path.join(PRICE_DATA, "**/**/*.parquet"))
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive= True)

parquet_files

['../../05_src/data/prices/NXJ/NXJ_2007/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2007/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2009/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2009/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2008/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2008/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2006/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2006/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2012/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2012/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2015/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2015/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2014/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2014/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2013/part.0.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2013/part.1.parquet',
 '../../05_src/data/prices/NXJ/NXJ_2004/part.0.parquet',
 '../../05_src/data/prices/NXJ/

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [39]:
# Write your code below.


dd_prices = (dd.
         read_parquet(parquet_files)
         .set_index("ticker"))

dd_prices



,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=90,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ABIO,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...


In [5]:
dd_prices_step2 = dd_prices.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

dd_prices_step2

/var/folders/lk/v5dfvh9d0799xjl8_99hhnhw0000gn/T/ipykernel_57379/1232178949.py:1: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_prices_step2 = dd_prices.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1
npartitions=90,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64
ABIO,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...


In [ ]:
# dd_prices_step2 = dd_prices.groupby('ticker', group_keys=False).apply(
#     lambda x: x.assign(Close_lag_1 = x['Close'].shift(1)).assign(Adj_Close_lag_1 = x['Adj Close'].shift(1)).assign(x['Close']/x['Close_lag_1'] - 1)
# )

In [ ]:
dd_prices_step3 = dd_prices_step2.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

dd_prices_step3

/var/folders/lk/v5dfvh9d0799xjl8_99hhnhw0000gn/T/ipykernel_57379/3278129226.py:1: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_prices_step3 = dd_prices_step2.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1
npartitions=90,,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64
ABIO,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...


In [15]:
dd_returns = dd_prices_step3.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

dd_returns

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns
npartitions=90,,,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64
ABIO,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
# dd_feat = dd_returns.assign(
#     Range = dd_returns['High'] - dd_returns['Low']
# )

# dd_feat

# dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Range
ticker,,,,,,,,,,,,,
A,2014-01-02,40.844063,40.844063,40.164520,40.207439,37.752617,2678800.0,A.csv,2014,NaN,NaN,NaN,0.679543
A,2014-01-03,40.336193,41.022888,40.243206,40.715309,38.229481,2609600.0,A.csv,2014,40.207439,37.752617,0.012631,0.779682
A,2014-01-06,41.058655,41.273247,40.457798,40.515022,38.041428,2484600.0,A.csv,2014,40.715309,38.229481,-0.004919,0.815449
A,2014-01-07,40.736767,41.223175,40.722462,41.094421,38.585449,2045500.0,A.csv,2014,40.515022,38.041428,0.014301,0.500713
A,2014-01-08,41.008583,41.874107,40.894135,41.766811,39.216789,3717900.0,A.csv,2014,41.094421,38.585449,0.016362,0.979973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2005-06-27,14.080000,14.330000,13.660000,13.730000,12.574939,273200.0,ZEUS.csv,2005,14.170000,12.977926,-0.031052,0.670000
ZEUS,2005-06-28,13.830000,14.360000,13.830000,14.150000,12.959603,290100.0,ZEUS.csv,2005,13.730000,12.574939,0.030590,0.530000
ZEUS,2005-06-29,14.200000,14.250000,13.650000,13.840000,12.675684,145400.0,ZEUS.csv,2005,14.150000,12.959603,-0.021908,0.600000


In [ ]:
dd_feat = dd_returns.assign(
    Range = lambda x: x['High'] - x['Low']
)

dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Range
npartitions=90,,,,,,,,,,,,,
A,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
ABIO,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [16]:
# Write your code below.

pd_feat = dd_feat.compute()
pd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Range
ticker,,,,,,,,,,,,,
A,2014-01-02,40.844063,40.844063,40.164520,40.207439,37.752617,2678800.0,A.csv,2014,NaN,NaN,NaN,0.679543
A,2014-01-03,40.336193,41.022888,40.243206,40.715309,38.229481,2609600.0,A.csv,2014,40.207439,37.752617,0.012631,0.779682
A,2014-01-06,41.058655,41.273247,40.457798,40.515022,38.041428,2484600.0,A.csv,2014,40.715309,38.229481,-0.004919,0.815449
A,2014-01-07,40.736767,41.223175,40.722462,41.094421,38.585449,2045500.0,A.csv,2014,40.515022,38.041428,0.014301,0.500713
A,2014-01-08,41.008583,41.874107,40.894135,41.766811,39.216789,3717900.0,A.csv,2014,41.094421,38.585449,0.016362,0.979973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZEUS,2005-06-27,14.080000,14.330000,13.660000,13.730000,12.574939,273200.0,ZEUS.csv,2005,14.170000,12.977926,-0.031052,0.670000
ZEUS,2005-06-28,13.830000,14.360000,13.830000,14.150000,12.959603,290100.0,ZEUS.csv,2005,13.730000,12.574939,0.030590,0.530000
ZEUS,2005-06-29,14.200000,14.250000,13.650000,13.840000,12.675684,145400.0,ZEUS.csv,2005,14.150000,12.959603,-0.021908,0.600000


In [11]:
rolling = pd_feat['Returns'].rolling(10).mean()
rolling

ticker
A            NaN
A            NaN
A            NaN
A            NaN
A            NaN
          ...   
ZEUS    0.001594
ZEUS   -0.006110
ZEUS    0.003339
ZEUS    0.010365
ZEUS    0.005329
Name: Returns, Length: 313227, dtype: float64

In [12]:
pd_feat['Rolling'] = rolling
pd_feat.head(30)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Range,Rolling
ticker,,,,,,,,,,,,,,
A,1999-11-18,32.546494,35.765381,28.612303,31.473534,27.068665,62546300.0,A.csv,1999,NaN,NaN,NaN,7.153078,NaN
A,1999-11-19,30.713520,30.758226,28.478184,28.880543,24.838577,15234100.0,A.csv,1999,31.473534,27.068665,-0.082386,2.280043,NaN
A,1999-11-22,29.551144,31.473534,28.657009,31.473534,27.068665,6577800.0,A.csv,1999,28.880543,24.838577,0.089783,2.816525,NaN
A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.607880,5975600.0,A.csv,1999,31.473534,27.068665,-0.090909,2.592991,NaN
A,1999-11-24,28.701717,29.998211,28.612303,29.372318,25.261524,4843200.0,A.csv,1999,28.612303,24.607880,0.026563,1.385908,NaN
A,1999-11-26,29.238197,29.685265,29.148785,29.461731,25.338428,1729400.0,A.csv,1999,29.372318,25.261524,0.003044,0.536480,NaN
A,1999-11-29,29.327610,30.355865,29.014664,30.132332,25.915169,4074700.0,A.csv,1999,29.461731,25.338428,0.022762,1.341202,NaN
A,1999-11-30,30.042919,30.713520,29.282904,30.177038,25.953619,4310000.0,A.csv,1999,30.132332,25.915169,0.001484,1.430616,NaN
A,1999-12-01,30.177038,31.071173,29.953505,30.713520,26.415012,2957300.0,A.csv,1999,30.177038,25.953619,0.017778,1.117668,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

The dask dataframe will not report values, only the overall schema, because of its "lazy execution".  In order to check the daily returns values and check that the calculation is correct, a pandas dataframe was more appropriate.

(Also, despite quite a bit of web searching, I still don't even know how to compute a rolling mean in dask. The .rolling(10).mean() functions described above seems specific to pandas dataframes. (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html). While I'm sure this task is technically possible in dask, most coders online seem to recommend converting dask to pandas for this calculation.)

In [13]:
# Question for learning support: How would we do this in dask?

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.